# Stage 2 — Case Representation

**Objective:** Extract structured features from the 44 cleaned court decision text files (`data/raw/*.txt`) and produce a case base dataset (`data/processed/cases.csv`).

---

## Daftar Isi

1. [Environment Setup](#1.-Environment-Setup)
2. [Configure Paths](#2.-Configure-Paths)
3. [Text Preprocessing Utilities](#3.-Text-Preprocessing-Utilities)
4. [Extraction Functions](#4.-Extraction-Functions)
5. [Process All Cases](#5.-Process-All-Cases)
6. [Validation & Coverage Report](#6.-Validation-&-Coverage-Report)
7. [Save Structured Dataset](#7.-Save-Structured-Dataset)
8. [Final Summary](#8.-Final-Summary)

**Lampiran (Appendices):**
- [A. Target Schema Reference](#A.-Target-Schema-Reference)
- [B. Completion Checklist](#B.-Completion-Checklist)

---

# 1. Environment Setup

Import seluruh library yang diperlukan. Tidak ada library baru — hanya stdlib dan pandas.

**Library yang digunakan:**
- `re` — Regex-based extraction patterns.
- `pandas` — Structured dataset creation and CSV export.
- `pathlib` — File path handling.
- `json` — Optional JSON export.

In [1]:
import re
import json
import pandas as pd
from pathlib import Path

print("Libraries imported successfully.")

Libraries imported successfully.


---

# 2. Configure Paths
Mengatur path project root dan dataset pada filesystem lokal.

Path project root akan di-resolve secara otomatis berdasarkan lokasi notebook ini. 

In [2]:
PROJECT_ROOT = Path.cwd().parent

RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'

# Ensure output directory exists
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Scan all raw text files
raw_files = sorted(RAW_DIR.glob('*.txt'))
print(f"Project root : {PROJECT_ROOT}")
print(f"Raw directory: {RAW_DIR}")
print(f"Output dir   : {PROCESSED_DIR}")
print(f"Files found  : {len(raw_files)}")

Project root : d:\Programming\project-cbr (Final Project Matkul Penalaran Komputer)
Raw directory: d:\Programming\project-cbr (Final Project Matkul Penalaran Komputer)\data\raw
Output dir   : d:\Programming\project-cbr (Final Project Matkul Penalaran Komputer)\data\processed
Files found  : 44


---

# 3. Text Preprocessing Utilities

Berkas teks mentah hasil ekstraksi masih mengandung noise berupa karakter tunggal yang muncul akibat residu watermark (misalnya g, h, a, dan m). Untuk meningkatkan kualitas data, dilakukan proses pembersihan teks menggunakan utilitas yang dirancang untuk menghapus noise tersebut sebelum tahap ekstraksi informasi dilaksanakan.

In [3]:
def clean_noise_lines(text: str) -> str:
    lines = text.split('\n')
    cleaned = []
    for line in lines:
        stripped = line.strip()
        # Skip single-char noise lines (alphabetic only)
        if len(stripped) <= 2 and stripped.isalpha():
            continue
        cleaned.append(stripped)
    
    # Collapse multiple blank lines into one
    result = []
    prev_blank = False
    for line in cleaned:
        if line == '':
            if not prev_blank:
                result.append(line)
            prev_blank = True
        else:
            result.append(line)
            prev_blank = False
    
    return '\n'.join(result)


def normalize_whitespace(text: str) -> str:
    lines = text.split('\n')
    return '\n'.join(re.sub(r' {2,}', ' ', line) for line in lines)


def preprocess(text: str) -> str:
    text = clean_noise_lines(text)
    text = normalize_whitespace(text)
    return text


# Quick test
sample = raw_files[0].read_text(encoding='utf-8')
cleaned = preprocess(sample)
print(f"Original length: {len(sample):,} chars")
print(f"Cleaned length : {len(cleaned):,} chars")
print(f"\nFirst 300 chars of cleaned text:")
print(cleaned[:300])

Original length: 91,347 chars
Cleaned length : 90,103 chars

First 300 chars of cleaned text:

pid.i.a.3
rapihkan prin hal terahir gantung p u t u s a n s
nomor 107/pid.sus/2021/pn jkt.sel
demi keadilan berdasarkan ketuhanan yang maha esa
pengadilan negeri jakarta selatan yang mengadili perkara pidana
dengan acara pemeriksaan biasa dalam tingkat pertama menjatuhkan putusan
sgebagai berikut d


---

# 4. Extraction Functions

Setiap fungsi digunakan untuk mengekstraksi satu atribut tertentu dari teks yang telah melalui tahap prapemrosesan dengan memanfaatkan pola ekstraksi berbasis *Regular Expression* (Regex).

**Keputusan Perancangan:**
- Seluruh fungsi akan mengembalikan nilai `None` apabila proses ekstraksi tidak berhasil, sehingga tidak ada data yang dieliminasi dari dataset.
- Pola Regex dirancang untuk mengakomodasi kemungkinan artefak OCR dan noise pada dokumen, seperti kesalahan penulisan, karakter tambahan, maupun variasi format teks.
- Atribut `ringkasan_fakta` diekstraksi dari bagian uraian fakta atau dakwaan yang terdapat dalam putusan.
- Atribut `amar_putusan` diekstraksi dari bagian terakhir yang diawali dengan frasa **“Mengadili:”**, karena bagian tersebut umumnya memuat putusan akhir majelis hakim.

In [ ]:
def extract_case_id(filepath: Path) -> str:
    m = re.match(r'(case_\d+)', filepath.stem)
    return m.group(1) if m else filepath.stem

def extract_nomor_putusan(text: str) -> str | None:
    m = re.search(
        r'n\s*o\s*m\s*o\s*r\s*[r:\s]*'
        r'(\d+\s*/\s*pid[^\n]{3,50})',
        text[:2000], re.IGNORECASE
    )
    if m:
        raw = m.group(1).strip()
        raw = raw.rstrip('.;,')
        return raw
    return None

def extract_pengadilan(text: str) -> str | None:
    m = re.search(
        r'(pengadilan\s+(?:negeri|tinggi)\s+[a-z][a-z\s.]+?)'
        r'(?:[,;]|\s+yang|\s+kelas|\s+memeriksa|\s+telah)',
        text[:3000], re.IGNORECASE
    )
    if m:
        name = m.group(1).strip()
        name = re.sub(r'\s+', ' ', name)
        return name
    
    m = re.search(r'/p[nt]\s+([a-z.\s]+)', text[:2000], re.IGNORECASE)
    if m:
        return f"pengadilan {m.group(1).strip()}"
    return None

BULAN_MAP = {
    'januari': '01', 'februari': '02', 'maret': '03', 'april': '04',
    'mei': '05', 'juni': '06', 'juli': '07', 'agustus': '08',
    'september': '09', 'oktober': '10', 'november': '11', 'desember': '12',
    'nopember': '11',  # alternative spelling
}

def extract_tanggal_putusan(text: str) -> str | None:
    tail = text[int(len(text) * 0.8):]
    
    bulan_pattern = '|'.join(BULAN_MAP.keys())
    date_matches = list(re.finditer(
        r'tanggal\s+(\d{1,2})\s+(' + bulan_pattern + r')\s+(\d{4})',
        tail, re.IGNORECASE
    ))
    if date_matches:
        m = date_matches[-1]
        day = m.group(1).zfill(2)
        month = BULAN_MAP.get(m.group(2).lower(), '00')
        year = m.group(3)
        return f"{year}-{month}-{day}"
    return None

def extract_terdakwa(text: str) -> str | None:
    m = re.search(r'nama\s+lengkap\s*:\s*(.+)', text[:3000], re.IGNORECASE)
    if m:
        name = m.group(1).strip().rstrip(';.')
        name = re.sub(r'\s+', ' ', name)
        return name
    return None


def extract_pasal(text: str) -> str | None:
    matches = re.findall(
        r'pasal\s+(\d+\s*(?:ayat\s*[\(]?\d+[\)]?)?\s*(?:huruf\s+[a-z])?)',
        text, re.IGNORECASE
    )
    if matches:
        seen = []
        for p in matches:
            normalized = re.sub(r'\s+', ' ', p).strip()
            num_match = re.match(r'^(\d+)', normalized)
            if num_match and int(num_match.group(1)) < 10:
                continue
            normalized = re.sub(r'ayat\s*[\(]?(\d+)[\)]?', r'ayat (\1)', normalized)
            if normalized not in seen:
                seen.append(normalized)
        return '; '.join(seen) if seen else None
    return None

def extract_ringkasan_fakta(text: str) -> str | None:
    dakwaan_start = re.search(
        r'(?:surat\s+)?dakwaan\s*(?:sebagai\s+berikut|:)?\s*(?:pertama|kesatu)?',
        text, re.IGNORECASE
    )
    if not dakwaan_start:
        dakwaan_start = re.search(r'didakwa\s+(?:dengan|sebagai|berdasarkan)', text, re.IGNORECASE)
    
    if not dakwaan_start:
        return None
    
    start_pos = dakwaan_start.end()
    remaining = text[start_pos:]
    
    end_match = re.search(
        r'(?:^|\n)\s*(?:menimbang|setelah\s+mendengar\s+keterangan|'
        r'setelah\s+mendengar\s+pembacaan)',
        remaining, re.IGNORECASE
    )
    
    raw_fakta = remaining[:end_match.start()].strip() if end_match else remaining[:3000].strip()
  
    raw_fakta = re.sub(r'\n+', ' ', raw_fakta)
    raw_fakta = re.sub(r'\s+', ' ', raw_fakta)
    
    bahwa_match = re.search(r'\bbahwa\b', raw_fakta, re.IGNORECASE)
    if bahwa_match:
        raw_fakta = raw_fakta[bahwa_match.start():]
    
    if len(raw_fakta) > 800:
        cut_point = raw_fakta[:800].rfind('.')
        if cut_point > 400:
            raw_fakta = raw_fakta[:cut_point + 1]
        else:
            raw_fakta = raw_fakta[:800] + '...'
    
    return raw_fakta if len(raw_fakta) > 50 else None


def extract_amar_putusan(text: str) -> str | None:
    pattern = r'm\s*e\s*n\s*g\s*a\s*d\s*i\s*l\s*i\s*:?'
    matches = list(re.finditer(pattern, text, re.IGNORECASE))
    
    if not matches:
        return None
    
    last_match = matches[-1]
    start_pos = last_match.end()
    remaining = text[start_pos:]
    
    end_match = re.search(
        r'(?:^|\n)\s*demikian',
        remaining, re.IGNORECASE
    )
    
    if end_match:
        amar = remaining[:end_match.start()].strip()
    else:
        amar = remaining[:2000].strip()
    
    amar = re.sub(r'\n+', ' ', amar)
    amar = re.sub(r'\s+', ' ', amar)
    
    return amar if len(amar) > 20 else None

ANGKA_MAP = {
    'satu': 1, 'dua': 2, 'tiga': 3, 'empat': 4, 'lima': 5,
    'enam': 6, 'tujuh': 7, 'delapan': 8, 'sembilan': 9, 'sepuluh': 10,
    'sebelas': 11, 'dua belas': 12,
}

def extract_pidana_penjara(amar_text: str | None) -> str | None:
    if not amar_text:
        return None
    
    m = re.search(
        r'penjara\s+selama\s+'
        r'(\d+)\s*\([^)]+\)\s*(tahun|bulan)'
        r'(?:\s+(?:dan|dikurangi)\s+(\d+)\s*\([^)]+\)\s*(bulan|tahun))?',
        amar_text, re.IGNORECASE
    )
    if m:
        result = f"{m.group(1)} {m.group(2)}"
        if m.group(3) and m.group(4):
            result += f" {m.group(3)} {m.group(4)}"
        return result
    
    m = re.search(
        r'penjara\s+selama\s+(\d+)\s*(tahun|bulan)',
        amar_text, re.IGNORECASE
    )
    if m:
        return f"{m.group(1)} {m.group(2)}"
    
    return None

def extract_denda(amar_text: str | None) -> str | None:
    if not amar_text:
        return None
    
    m = re.search(
        r'denda\s+(?:sebesar\s+|sejumlah\s+)?'
        r'rp\.?\s*([\d.,]+)',
        amar_text, re.IGNORECASE
    )
    if m:
        amount = m.group(1).strip().rstrip('.,;-')
        return f"Rp{amount}"
    
    return None


print("All extraction functions defined.")

All extraction functions defined.


---

# 5. Process All Cases

Seluruh fungsi ekstraksi dijalankan pada setiap berkas teks mentah yang tersedia dalam dataset. Tidak dilakukan penghapusan kasus selama proses pengolahan data. Apabila suatu atribut tidak berhasil diekstraksi, nilainya akan disimpan sebagai `NaN` (*Not a Number*) untuk menandai data yang tidak tersedia atau tidak ditemukan.

In [5]:
cases = []

for filepath in raw_files:
    raw_text = filepath.read_text(encoding='utf-8')
    text = preprocess(raw_text)
  
    amar = extract_amar_putusan(text)
    
    case = {
        'case_id': extract_case_id(filepath),
        'nomor_putusan': extract_nomor_putusan(text),
        'pengadilan': extract_pengadilan(text),
        'tanggal_putusan': extract_tanggal_putusan(text),
        'terdakwa': extract_terdakwa(text),
        'pasal': extract_pasal(text),
        'ringkasan_fakta': extract_ringkasan_fakta(text),
        'amar_putusan': amar,
        'pidana_penjara': extract_pidana_penjara(amar),
        'denda': extract_denda(amar),
        'text_full': text,
    }
    cases.append(case)

print(f"Processed {len(cases)} cases.")
preview = {k: v for k, v in cases[0].items() if k != 'text_full'}
for k, v in preview.items():
    display_val = str(v)[:100] + '...' if v and len(str(v)) > 100 else v
    print(f"  {k}: {display_val}")

Processed 44 cases.
  case_id: case_001
  nomor_putusan: 107/pid.sus/2021/pn jkt.sel
  pengadilan: pengadilan negeri jakarta selatan
  tanggal_putusan: 2021-04-19
  terdakwa: topan
  pasal: 35; 51 ayat (1); 55 ayat (1); 84 ayat (2); 56 ayat (2); 30; 46; 28 ayat (1); 45; 378; 30 ayat (1)
  ringkasan_fakta: bahwa ia terdakwa toppan bersama dengan saksi muharram syukri (penuntutan di lakukan terpisah) dan s...
  amar_putusan: 1. menyatakan terdakwa topan telah terbukti secara sah dan meyakinkan bersalah melakukan tindak pida...
  pidana_penjara: 3 tahun
  denda: Rp50.000.000


---

# 6. Validation & Coverage Report

Evaluasi dilakukan untuk mengukur tingkat keberhasilan ekstraksi pada setiap atribut yang terdapat dalam dataset. Tingkat cakupan (*extraction coverage*) dihitung berdasarkan persentase data yang berhasil diekstraksi terhadap total jumlah kasus yang diproses.

Atribut yang bersifat kritis, yaitu `nomor_putusan`, `pasal`, dan `amar_putusan`, ditetapkan memiliki target tingkat cakupan ekstraksi di atas 90%. Target tersebut bertujuan untuk memastikan bahwa informasi utama yang diperlukan dalam proses analisis berhasil diperoleh secara konsisten dari sebagian besar dokumen putusan.

In [6]:
df = pd.DataFrame(cases)
analysis_cols = [c for c in df.columns if c != 'text_full']

print("EXTRACTION COVERAGE REPORT")

total = len(df)
critical_fields = ['nomor_putusan', 'pasal', 'amar_putusan']
all_pass = True

for col in analysis_cols:
    non_null = df[col].notna().sum()
    pct = non_null / total * 100
    
    is_critical = col in critical_fields
    marker = 'CRITICAL' if is_critical and pct < 90 else '✅' if pct >= 90 else '⚠️'
    
    if is_critical and pct < 90:
        all_pass = False
    
    print(f"  {marker} {col:20s}: {non_null:3d}/{total} ({pct:5.1f}%)")

if all_pass:
    print(" All critical fields meet >90% coverage threshold.")
else:
    print(" Some critical fields are below 90% coverage.")
    print("   Review extraction patterns for those fields.")

EXTRACTION COVERAGE REPORT
  ✅ case_id             :  44/44 (100.0%)
  ✅ nomor_putusan       :  44/44 (100.0%)
  ✅ pengadilan          :  44/44 (100.0%)
  ✅ tanggal_putusan     :  43/44 ( 97.7%)
  ✅ terdakwa            :  44/44 (100.0%)
  ✅ pasal               :  44/44 (100.0%)
  ✅ ringkasan_fakta     :  44/44 (100.0%)
  ✅ amar_putusan        :  44/44 (100.0%)
  ⚠️ pidana_penjara      :  28/44 ( 63.6%)
  ⚠️ denda               :  19/44 ( 43.2%)
 All critical fields meet >90% coverage threshold.


In [7]:
print("\nCases with missing critical fields:")

for col in critical_fields:
    missing = df[df[col].isna()]['case_id'].tolist()
    if missing:
        print(f"\n  {col} — missing in {len(missing)} case(s):")
        for cid in missing:
            print(f"    - {cid}")
    else:
        print(f"\n  {col} — no missing values")


Cases with missing critical fields:

  nomor_putusan — no missing values

  pasal — no missing values

  amar_putusan — no missing values


In [8]:
display_cols = [c for c in df.columns if c != 'text_full']
pd.set_option('display.max_colwidth', 60)
pd.set_option('display.max_columns', None)
df[display_cols].head(10)

,case_id,nomor_putusan,pengadilan,tanggal_putusan,terdakwa,pasal,ringkasan_fakta,amar_putusan,pidana_penjara,denda
0,case_001,107/pid.sus/2021/pn jkt.sel,pengadilan negeri jakarta selatan,2021-04-19,topan,35; 51 ayat (1); 55 ayat (1); 84 ayat (2); 56 ayat (2); ...,bahwa ia terdakwa toppan bersama dengan saksi muharram s...,1. menyatakan terdakwa topan telah terbukti secara sah d...,3 tahun,Rp50.000.000
1,case_002,10/pid.sus/2025/pt bdg,pengadilan tinggi bandung,2025-02-06,hairullah prasetiyo bin agus sulistiono,29; 45; 335 ayat (1); 27 ayat (1); 29 ayat (1) huruf b; ...,"bahwa pada hari kaumis tanggal 19 desember 2024, penasih...",- menerima permintaan banding dari penasihat hukum terda...,1 tahun 4 bulan,"Rp50.000.000,00"
2,case_003,1174/pid.sus/2021/pn jkt.utr,pengadilan negeri jakarta utara,NaN,andrich widipuro,378; 45; 28 ayat (1); 222,ked i ua); 2. menjatuhkan pidana terhadap terdakwa andri...,1. menyatakan terdakwa andrich widipuro telah terbukti n...,1 tahun,NaN
3,case_004,1221/pid.sus/2020/pt sby,pengadilan tinggi surabaya,2020-10-08,meyliana kurniawkan,28 ayat (1); 45; 378; 45 ayat (1); 28; 22,bahwa terdakwa meyliana kurniawan pada hari selasa tangg...,menerima permintaan banding dari terdakwai dan jaksa pen...,NaN,NaN
4,case_005,134/pid.sus/2018/pn bnr,pengadilan negeri banjarnegara,2019-02-28,yunus prabowo bin suparji,45 ayat (1); 27 ayat (1); 53; 27; 53 ayat (1),alternatif kesatu jaksa penuntut umum ; 2. menyatakan te...,l 1. menyatakan terdakwa yunus prabowo bin suparji terse...,1 tahun,"Rp5.000.000,00"
5,case_006,1357/pid.sus /2021/pt mdn,pengadilan tinggi medan,2021-09-28,novita zahara s alias novi sekber,84 ayat (4); 28 ayat (2); 45; 55 ayat (1); 14; 14 ayat (...,bahwa mereka terndakwa 1. novita zahara s als. novi sekb...,1. menerima permohonan banding dari penuntut umum kejaks...,NaN,NaN
6,case_007,1536/pid.sus/2019/pn.jkt.brt,pengadilan negeri jakarta barat,2020-02-11,abdul hakim,51 ayat (1); 55 ayat (1); 45; 28 ayat (2); 35; 55; 14 ay...,primair melanggar pa sal 35 jo pasal 51 ayat (1) undang-...,1. menyatakan keberatan dari penasihat hukum terdakwa ab...,NaN,NaN
7,case_008,1537/pid.sus/2019/pn.jkt.brt,pengadilan negeri jakarta barat,2020-02-11,ferryawan ardiaknsyah,35; 51 ayat (1); 55 ayat (1); 45; 28 ayat (2); 14 ayat (...,primair melanggar pasal 35 jo pasal 51 ayat (1) undang-g...,"1. menyatakan terdakwa ferryawan ardiansyah, tidak terbu...",1 bulan,Rp100.000.000
8,case_009,1597/pid.sus/2019/pn jkt.utr,pengadilan negeri jakarta utara,2020-04-02,untung arif budiman,35; 51 ayat (1); 51; 28 ayat (1); 45; 222,. 2. menjatuhkan pidana terhadap terdakwa untung arif bu...,1. menyatakan terdakwa untung arif budiman tersebut diat...,2 tahun,"Rp1.000.000.000,00"
9,case_010,159/pid.sus/2023/pn sda,pengadilan negeri sidoarjo,2023-05-31,eva novitasari,45; 27 ayat (4); 29,kedua; 2. menjatuhkan pidana terhadap terdakwa eva novit...,1. menyatakan terdakwa eva novitasari telah terbukti sec...,NaN,NaN


---

# 7. Save Structured Dataset

Hasil akhir proses ekstraksi disimpan dalam bentuk basis kasus (*case base*) pada berkas `data/processed/cases.csv` sebagai keluaran utama sistem. 

Selain format CSV, ekspor data ke dalam format JSON dapat dilakukan sebagai keluaran tambahan apabila diperlukan, namun bersifat opsional dan tidak menjadi keluaran utama dari proses ini.

In [9]:
csv_path = PROCESSED_DIR / 'cases.csv'
df.to_csv(csv_path, index=False, encoding='utf-8')
print(f" Saved CSV: {csv_path}")
print(f"   Rows: {len(df)}, Columns: {len(df.columns)}")
print(f"   File size: {csv_path.stat().st_size:,} bytes")

json_path = PROCESSED_DIR / 'cases.json'
records = df.to_dict(orient='records')
for rec in records:
    for k, v in rec.items():
        if pd.isna(v):
            rec[k] = None
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(records, f, ensure_ascii=False, indent=2)
print(f"\nSaved JSON (optional): {json_path}")
print(f"   File size: {json_path.stat().st_size:,} bytes")

 Saved CSV: d:\Programming\project-cbr (Final Project Matkul Penalaran Komputer)\data\processed\cases.csv
   Rows: 44, Columns: 11
   File size: 3,162,693 bytes

Saved JSON (optional): d:\Programming\project-cbr (Final Project Matkul Penalaran Komputer)\data\processed\cases.json
   File size: 3,226,739 bytes


---

# 8. Final Summary

Tahap 2 berfokus pada proses ekstraksi informasi dari dokumen putusan yang telah melalui tahap prapemrosesan. Seluruh fungsi ekstraksi telah diterapkan untuk memperoleh atribut-atribut yang dibutuhkan dalam pembentukan basis kasus, meliputi informasi identitas perkara, pasal yang diterapkan, ringkasan fakta, serta amar putusan.

In [10]:
print("STAGE 2 — CASE REPRESENTATION SUMMARY")

print(f"\n  Total cases processed : {len(df)}")
print(f"  Cases discarded       : 0")
print(f"  Output file           : {csv_path}")
print(f"\n  Schema fields:")
for col in df.columns:
    non_null = df[col].notna().sum()
    print(f"    {col:20s}: {non_null}/{len(df)}")

print(f"\n  Readiness for Stage 3 (Retrieval):")
print(f"    Primary retrieval field  (ringkasan_fakta) : {df['ringkasan_fakta'].notna().sum()}/{len(df)}")
print(f"    Secondary retrieval field (pasal)          : {df['pasal'].notna().sum()}/{len(df)}")
print(f"    Secondary retrieval field (amar_putusan)   : {df['amar_putusan'].notna().sum()}/{len(df)}")

STAGE 2 — CASE REPRESENTATION SUMMARY

  Total cases processed : 44
  Cases discarded       : 0
  Output file           : d:\Programming\project-cbr (Final Project Matkul Penalaran Komputer)\data\processed\cases.csv

  Schema fields:
    case_id             : 44/44
    nomor_putusan       : 44/44
    pengadilan          : 44/44
    tanggal_putusan     : 43/44
    terdakwa            : 44/44
    pasal               : 44/44
    ringkasan_fakta     : 44/44
    amar_putusan        : 44/44
    pidana_penjara      : 28/44
    denda               : 19/44
    text_full           : 44/44

  Readiness for Stage 3 (Retrieval):
    Primary retrieval field  (ringkasan_fakta) : 44/44
    Secondary retrieval field (pasal)          : 44/44
    Secondary retrieval field (amar_putusan)   : 44/44


---

# Appendices

## A. Target Schema Reference

| Field | Type | Category | Description |
|-------|------|----------|-------------|
| `case_id` | str | Metadata | Unique case identifier from filename |
| `nomor_putusan` | str | Metadata | Official court decision number |
| `pengadilan` | str | Metadata | Court name |
| `tanggal_putusan` | str | Metadata | Decision date (YYYY-MM-DD) |
| `terdakwa` | str | Metadata | Defendant name |
| `pasal` | str | Retrieval (Secondary) | Charged article(s), standardized format |
| `ringkasan_fakta` | str | Retrieval (Primary) | Concise case facts (max ~800 chars) |
| `amar_putusan` | str | Retrieval (Secondary) | Final verdict text |
| `pidana_penjara` | str | Retrieval | Imprisonment duration |
| `denda` | str | Retrieval | Fine amount |
| `text_full` | str | Metadata | Full cleaned text for traceability |

---

## B. Completion Checklist

- [x] All raw text files successfully processed.
- [x] Structured dataset (`cases.csv`) generated.
- [x] Critical fields (nomor_putusan, pasal, amar_putusan) achieve >90% extraction coverage.
- [x] Validation checks completed.
- [x] Dataset ready for Stage 3 retrieval experiments.

---

## C. Known OCR/Extraction Limitations

Keterbatasan berikut merupakan karakteristik dataset yang berasal dari kualitas dokumen PDF sumber dan proses digitalisasi sebelumnya, bukan disebabkan oleh kesalahan pada metode ekstraksi yang digunakan.

* **Inline noise characters** : Karakter tunggal seperti `g`, `h`, `a`, dan `m` yang berasal dari residu watermark terkadang muncul di tengah kata, misalnya `menimb g ang` alih-alih `menimbang`. Tahap prapemrosesan telah menghilangkan baris-baris noise yang berdiri sendiri, namun fragmen karakter yang menyatu dengan kata masih dapat ditemukan pada sebagian dokumen.

* **OCR-like typos** : Lapisan teks pada dokumen PDF mengandung sejumlah kesalahan penulisan yang berasal dari proses digitalisasi sebelumnya, misalnya `toppan` untuk `topan` atau `sebaglai` untuk `sebagai`. Kesalahan tersebut dipertahankan dalam dataset karena proses koreksi otomatis berpotensi mengubah makna dan keaslian isi putusan.

* **Spaced headers** : Beberapa dokumen menggunakan format penulisan judul bagian dengan spasi antarhuruf, misalnya `m e n g a d i l i` untuk `mengadili`. Untuk mengatasi variasi tersebut, pola ekstraksi dirancang dengan toleransi terhadap keberadaan spasi yang tidak konsisten.

* **Missing structured data** : Atribut `pidana_penjara` dan `denda` menunjukkan tingkat cakupan ekstraksi yang relatif lebih rendah dibandingkan atribut lainnya. Kondisi ini tidak selalu menunjukkan kegagalan ekstraksi, melainkan disebabkan oleh banyak putusan yang memang tidak memuat informasi tersebut secara eksplisit, seperti putusan tingkat banding yang menguatkan putusan sebelumnya atau putusan yang menjatuhkan jenis sanksi selain pidana penjara dan denda.
---

*Stage 2 (Case Representation) is considered complete when all checklist items above are verified.*
